In [2]:

import sys
import os
import time
import numpy as np

import MDAnalysis as mda
from MDAnalysis.analysis import align
from MDAnalysis.analysis import distances

import westpa
from westpa.analysis import Run

import matplotlib.pyplot as plt

/home/jonathan/anaconda3/envs/serpents/lib/python3.11/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [3]:
def tmd_query():
    segment_resis = [[77, 149], [192, 245], [298, 362], [988, 1034], [857, 889], [900, 942], [1094, 1154]]
    #print("color blue, " + " or ".join([f"resi {sr[0]}-{sr[1]}" for sr in segment_resis]))

    segment_resis_all = [i for sr in segment_resis for i in range(sr[0], sr[1]+1)]
    query = " or ".join([f"resid {sr}" for sr in segment_resis_all])

    #indices = frame.top.select(f"protein and ({query})")

    # print_indices = frame.top.select(f"protein and name CA and ({query})")
    # print("+".join([str(i+1) for i in print_indices]))
    return f"protein and ({query})"


def ref_prot_crd():
    ref_frame_path = '/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input.gro'
    ref_frame = mda.Universe(ref_frame_path)
    ref_sel = ref_frame.select_atoms(f"{tmd_query()} and name CA")#not name H*")
    lig_sel = ref_frame.select_atoms("resname LJP and not name H*")
    return ref_sel.positions, lig_sel.positions

prot_pos, lig_pos = ref_prot_crd()

In [4]:
#this barely needs to be a method but having the .h5 stuff compartmentalized is nice
def load_h5_pcs(h5path, miniter, maxiter):
    
    run = Run.open(h5path)

    #set maximum iteration automatically
    if maxiter == -1:
        maxiter = run.num_iterations

    pcs = [iteration.pcoords for iteration in run if (iteration.number >= miniter and iteration.number < maxiter)]

    return pcs

In [5]:
#specify input file

cftr_west = "/home/jonathan/Documents/grabelab/cftr/chloe-data"
cftr_refpc = "/home/jonathan/Documents/grabelab/cftr/refeaturization"

h5paths_names = [[f"{cftr_west}/wstp_cftr_1_degrabo/west-040925.h5", f"{cftr_refpc}/nonlip_glpg_1", "pyrazole-1", "blue"],
                  [f"{cftr_west}/wstp_cftr_2_wynton/west-040925.h5", f"{cftr_refpc}/nonlip_glpg_2", "pyrazole-2", "cyan"],
                  [f"{cftr_west}/wstp_lip_glpg_1/west-040925.h5", f"{cftr_refpc}/lip_glpg_1", "undecanol-1", "red"],
                  [f"{cftr_west}/wstp_lip_glpg_2/west-040925.h5", f"{cftr_refpc}/lip_glpg_2", "undecanol-2", "orange"]]

#westpa rounds to load
minround = 0
maxround = -1

run_ind = 0

water_coords_data_paths = ["/home/jonathan/Documents/grabelab/cftr/revisions/abbv-974-1"]
water_coords_data_path = water_coords_data_paths[run_ind]


In [6]:
pcs_all = load_h5_pcs(h5paths_names[run_ind][0], minround, maxround)

In [7]:
nbins = 51
binbounds = np.arange(0,nbins,1)
waters_by_bin = [[] for a in range(nbins)]

#loop over WE rounds
for r in range(1,1000,10):
    
    #get progress coordinates of the walkers, accounting for the occasional corrupted file
    walkers = np.load(f"{water_coords_data_path}/pc_data_round_{r}_obs_0_v1.npy")
    pcs_flat = pcs_all[r-1][:,-1].flatten()
    pcs = [pcs_flat[w] for w in walkers]

    bins = np.digitize(pcs, binbounds)

    #load water coordinates
    waters = np.load(f"{water_coords_data_path}/pc_data_round_{r}_obs_1_v1.npy")
    print(waters.shape)
    #get coordinates of waters in each bin
    for b, w in zip(bins, waters):
        waters_by_bin[b].append(w)

waters_by_bin = [np.stack(w) if len(w) > 0 else np.array([]) for w in waters_by_bin]

for i, w in enumerate(waters_by_bin):
    print(f"{i}: {w.shape[0]}")

(5246, 3)
(31391, 3)
(36341, 3)
(41282, 3)
(52771, 3)
(36794, 3)
(36693, 3)
(36832, 3)
(42140, 3)
(26575, 3)
(30922, 3)
(47693, 3)
(71931, 3)
(82986, 3)
(83207, 3)
(87481, 3)
(103617, 3)
(103864, 3)
(103118, 3)
(92835, 3)
(103510, 3)
(107625, 3)
(108324, 3)
(87670, 3)
(109113, 3)
(104272, 3)
(105273, 3)
(119523, 3)
(119773, 3)
(143403, 3)
(140173, 3)
(145003, 3)
(130224, 3)
(135313, 3)
(140793, 3)
(131569, 3)
(130905, 3)
(140800, 3)
(139675, 3)
(140559, 3)
(134711, 3)
(139694, 3)
(129526, 3)
(129192, 3)
(128920, 3)
(129686, 3)
(129642, 3)
(133452, 3)
(129171, 3)
(143112, 3)
(138756, 3)
(139849, 3)
(134310, 3)
(135064, 3)
(144057, 3)
(138915, 3)
(145279, 3)
(138232, 3)
(138405, 3)
(138170, 3)
(144179, 3)
(134723, 3)
(139297, 3)
(144641, 3)
(138699, 3)
(133407, 3)
(126941, 3)
(137262, 3)
(137420, 3)
(127249, 3)
(141361, 3)
(131787, 3)
(137484, 3)
(128571, 3)
(138766, 3)
(127200, 3)
(132026, 3)
(127177, 3)
(132209, 3)
(133119, 3)
(138589, 3)
(124763, 3)
(134240, 3)
(138282, 3)
(134223, 3)

In [9]:

def coords_to_pml(coords, output_file="pseudoatoms.pml", object_name="points"):
    """
    Generate a PyMOL .pml script that creates pseudoatoms at given coordinates.

    Parameters
    ----------
    coords : (N,3) numpy array
        Array of xyz coordinates
    output_file : str
        Output .pml filename
    object_name : str
        Name of the PyMOL object
    """

    if coords.ndim != 2 or coords.shape[1] != 3:
        raise ValueError("coords must be an Nx3 numpy array")

    with open(output_file, "w") as f:
        f.write(f"delete {object_name}\n")

        for i, (x, y, z) in enumerate(coords, start=1):
            f.write(
                f"pseudoatom {object_name}, pos=[{x:.3f},{y:.3f},{z:.3f}], "
                f"name=P{i}, resi={i}, chain=A\n"
            )

        f.write(f"show spheres, {object_name}\n")
        f.write(f"set sphere_scale, 0.3, {object_name}\n")

bin = 10

coords_to_pml(waters_by_bin[bin], f"/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/bin_{bin}_waters.pml")

# if __name__ == "__main__":

#     if len(sys.argv) < 3:
#         print("Usage: python coords_to_pml.py coords.npy output.pml")
#         sys.exit(1)

#     coords = np.load(sys.argv[1])
#     output_file = sys.argv[2]

#     coords_to_pml(coords, output_file)

In [84]:
import numpy as np
import plotly.graph_objects as go

# Example large dataset
# N = 200000
# coords = np.random.randn(N,3)
# x = coords[:,0]
# y = coords[:,1]
# z = coords[:,2]

mean_prot = np.mean(prot_pos, axis = 0)

bin = 5
interval=1

#print(waters_by_bin[bin])

x = waters_by_bin[bin][::interval,0]-mean_prot[0]
y = waters_by_bin[bin][::interval,1]-mean_prot[1]
z = waters_by_bin[bin][::interval,2]-mean_prot[2]

fig = go.Figure()

fig.add_trace(
    go.Scatter3d(
        x=x,
        y=y,
        z=z,
        mode="markers",
        hoverinfo="skip",
        marker=dict(
            size=3,
            opacity=0.7,
            color="red"#z,          # GPU color mapping
            #colorscale="Viridis"
        )
    )
)


if True:
    xp = prot_pos[:,0]-mean_prot[0]
    yp = prot_pos[:,1]-mean_prot[1]
    zp = prot_pos[:,2]-mean_prot[2]

    fig.add_trace(
        go.Scatter3d(
            x=xp,
            y=yp,
            z=zp,
            mode="markers",
            hoverinfo="skip",
            marker=dict(
                size=4,
                opacity=1,
                color="black"
            )
        )
    )

    xl = lig_pos[:,0]-mean_prot[0]
    yl = lig_pos[:,1]-mean_prot[1]
    zl = lig_pos[:,2]-mean_prot[2]

    fig.add_trace(
        go.Scatter3d(
            x=xl,
            y=yl,
            z=zl,
            mode="markers",
            hoverinfo="skip",
            marker=dict(
                size=4,
                opacity=1,
                color="blue"
            )
        )
    )

fig.update_layout(
    title="Large Interactive 3D Scatter",
    scene=dict(
        xaxis = dict(nticks=4, range=[-30,30],),
        yaxis = dict(nticks=4, range=[-30,30],),
        zaxis = dict(nticks=4, range=[-30,30],),
        aspectratio=dict(x=1, y=1, z=1),
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z"
    ),
    width=1000,
    height=1000,
    margin=dict(l=0,r=0,b=0,t=40),
    scene_camera=dict(
            eye=dict(x=3, y=3, z=3)
        )

)

fig.show(config={"scrollZoom": True})